# Walmart B2B — Node Level Cost Pricing

Orchestrator notebook for the full NLC pricing pipeline.

**Steps:**
1. Set parameters & flags
2. Load data
3. Run NLC model (two-pass: min_units=8, then 4)
4. Apply pricing rules
5. Build new DSV
6. Update tests tracker
7. Save outputs

**Optional steps** (toggle via flags):
- Rollback handling
- National price updates

**Post-upload** (run separately, ~3 hours after hybris upload):
- FTP response validation

# Params

In [ ]:
import pandas as pd

# ── Core parameters ─────────────────────────────────────────────────
date_str = pd.to_datetime('today').strftime('%Y-%m-%d')
test = False
save = True

# ── Optional steps ──────────────────────────────────────────────────
apply_rollbacks = True              # Remove rollback SKUs from NLC + apply RB prices to national
update_national_prices = False      # Override national prices from Excel file

# ── Margin test filter (set to None to include all) ─────────────────
margin_test_start_dates = None      # e.g. ["2026-03-12"]

# ── National price update config (only used if update_national_prices=True)
national_prices_path = None         # e.g. r"H:\...\Walmart B2B Item Report 2.20.26.xlsx"
national_prices_sheet = "National prices"
national_prices_skip_rows = 2

print(f"Date: {date_str}")
print(f"Test mode: {test}")
print(f"Apply rollbacks: {apply_rollbacks}")
print(f"Update national prices: {update_national_prices}")

# Setup

In [ ]:
import sys
import os
import logging

import numpy as np

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
)

from src.data.loader import DataLoader
from src.models.nlc_model import NLCModel
from src.rules.pricing_rules import PricingRulesEngine
from src.dsv.dsv_builder import DSVBuilder
from src.tracker.tracker_updater import TrackerUpdater

print(f"Project root: {project_root}")

# Step 1-2: Load data & run NLC model

In [ ]:
loader = DataLoader()
model = NLCModel(date_str=date_str)
model.load_data(loader)
df_output = model.run()

print(f"NLC output: {len(df_output)} SKU-Node rows")
display(df_output["Final target"].value_counts())
display(df_output["current_nlc_margin category"].value_counts())

## New SKU-Nodes + checks

In [ ]:
df_new = df_output[df_output["New NLC"] == "Yes"].copy()
print(f"New SKU-Nodes: {len(df_new)}")
display(df_new["Min units"].value_counts())
display(df_new["Brand code"].value_counts().head(10))
display(df_new["Identifier"].value_counts().head(10))

## Update current NLC margins & stock status

In [ ]:
df_current_tests = model.df_current_tests.copy()
print(f"Tracker entries: {len(df_current_tests)}")
display(df_current_tests["Final target"].value_counts())

# Step 3: Pricing rules

In [ ]:
engine = PricingRulesEngine(
    df_output, model.df_current_tests, date_str, test_mode=test
)

## Walmart margin split test update

In [ ]:
df_wm_split_dsv, df_wm_split_tracker = engine.get_wm_margin_split_updates()
print(f"Wm margin split DSV updates: {len(df_wm_split_dsv)}")
if len(df_wm_split_dsv) > 0:
    display(df_wm_split_dsv.head())

## Brands margin test update

In [ ]:
df_margin_dsv, df_margin_tracker = engine.get_margin_test_updates(
    start_dates=margin_test_start_dates
)
print(f"Margin test DSV updates: {len(df_margin_dsv)}")
if len(df_margin_dsv) > 0:
    display(df_margin_dsv.head())

## Low price updates

In [ ]:
df_low_dsv, df_low_tracker = engine.get_low_price_updates()
print(f"Low price DSV updates: {len(df_low_dsv)}")
if len(df_low_dsv) > 0:
    display(df_low_tracker["Category inventory"].value_counts())
    display(df_low_tracker["Min units"].value_counts())

## High price updates

In [ ]:
df_high_dsv, df_high_tracker = engine.get_high_price_updates()
print(f"High price DSV updates: {len(df_high_dsv)}")
if len(df_high_tracker) > 0:
    summary = (
        df_high_tracker.groupby("Final target")
        .agg(
            count=("Final target", "count"),
            avg_delta=("Final delta vs current %", "mean"),
        )
        .reset_index()
    )
    display(summary)

## New SKU-Nodes

In [ ]:
df_new_nodes_dsv, df_new_nodes_tracker = engine.get_new_sku_nodes()
print(f"New SKU-Nodes for DSV: {len(df_new_nodes_dsv)}")
if len(df_new_nodes_tracker) > 0:
    display(df_new_nodes_tracker["Final price category"].value_counts())

# Step 4: Build new DSV

In [ ]:
builder = DSVBuilder(
    df_curr_dsv_original=model.df_curr_dsv_original,
    today_str=date_str,
)

# Start from current DSV
df_dsv_start = builder._prepare_starting_dsv()
print(f"Starting DSV: {len(df_dsv_start)} rows")

## [Optional] Update national prices

Toggle: `update_national_prices` flag in Params cell.

In [ ]:
if update_national_prices:
    print(f"Updating national prices from: {national_prices_path}")
    df_dsv_start = builder.apply_national_price_updates(
        df_dsv_start,
        national_prices_path=national_prices_path,
        sheet_name=national_prices_sheet,
        skip_rows=national_prices_skip_rows,
    )
else:
    print("[Skipped] Update national prices")

## [Optional] Rollback handling

Toggle: `apply_rollbacks` flag in Params cell.

When enabled:
- Removes rollback SKUs from NLC rows
- Applies rollback prices to national rows

In [ ]:
if apply_rollbacks:
    print(f"Active rollbacks: {len(model.df_rollbacks)} rows, "
          f"{model.df_rollbacks['Product Code'].nunique()} unique SKUs")
    df_dsv_start = builder.apply_rollbacks(df_dsv_start, model.df_rollbacks)
else:
    print("[Skipped] Rollback handling")

## Apply NLC updates & build final DSV

In [ ]:
list_dsv_updates = [df_wm_split_dsv, df_margin_dsv, df_low_dsv, df_high_dsv]

df_new_dsv = builder.build_from(
    df_dsv_start,
    list_dsv_updates=list_dsv_updates,
    df_new_nodes=df_new_nodes_dsv,
)

print(f"Final DSV: {len(df_new_dsv)} rows")
df_new_dsv.head()

## DSV validation

In [ ]:
df_validation = builder.validate(df_new_dsv)

display(df_validation["Price change category"].value_counts())

df_changes = df_validation[df_validation["Price change category"] != "No change"]
df_changes["Is national?"] = np.where(
    df_changes["Source"] == "National", "Yes", "No"
)
display(df_changes["Is national?"].value_counts())
df_changes.head(10)

# Step 5: Save DSV

In [ ]:
if save:
    dsv_path = builder.save(df_new_dsv)
    print(f"DSV saved to: {dsv_path}")
else:
    print("[Skipped] DSV save (save=False)")

# Step 6: Update tracker

In [ ]:
updater = TrackerUpdater(model.df_current_tests, today_str=date_str)

# Refresh margins from model output
updater.update_margins(df_output)

# Append all new/updated entries
updater.append_entries([
    df_new_nodes_tracker,
    df_low_tracker,
    df_high_tracker,
    df_wm_split_tracker,
    df_margin_tracker,
])

print(f"Tracker: {len(updater.df_tracker)} total rows")
print(f"Expected: {len(model.df_current_tests) + len(df_new_nodes_tracker)}")

display(updater.df_tracker["Final target"].value_counts().sort_index())

In [ ]:
if save:
    tracker_path = updater.save()
    print(f"Tracker saved to: {tracker_path}")
else:
    print("[Skipped] Tracker save (save=False)")

# Cleanup

In [ ]:
loader.close()
print("Done.")

---

# Post-upload: FTP validation

Run this section **~3 hours after uploading** the DSV via hybris.

Downloads XML response files from Walmart FTP, parses ingestion status,
and generates a summary report. Alerts if failure rate exceeds 1.5%.

In [ ]:
from src.dsv.ftp_validator import FTPValidator

validator = FTPValidator(today_str=date_str)
n_files = validator.download_responses()
print(f"Downloaded {n_files} response files")

In [ ]:
if n_files > 0:
    df_results = validator.parse_responses()
    display(df_results["ingestionStatus"].value_counts())

    report_path = validator.generate_report(df_results)
    print(f"Report saved: {report_path}")
else:
    print("No response files yet. Try again later.")